# 01 — Análisis Exploratorio de Datos (EDA)

**Proyecto:** Predicción espacio-temporal de brotes de dengue en CABA  
**Sprint 3 / HU4:** Análisis exploratorio de patrones espacio-temporales  
**Autora:** Ing. Paola Andrea Blanco Blanco

---

## Objetivos de este notebook

1. Explorar series temporales de casos de dengue por semana epidemiológica
2. Visualizar distribución geográfica de casos por comuna de CABA
3. Calcular correlaciones entre variables climáticas y casos de dengue
4. Analizar autocorrelación espacial (comunas vecinas)
5. Identificar patrones de estacionalidad (temporada alta: dic-abr)

**Criterios de aceptación HU4:**
- [ ] Visualizaciones de series temporales por semana epidemiológica
- [ ] Mapas de calor por comuna y período
- [ ] Correlaciones clima-dengue calculadas
- [ ] Autocorrelación espacial entre comunas analizada
- [ ] Hallazgos documentados

## 0. Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import geopandas as gpd
import yaml
import warnings
warnings.filterwarnings('ignore')

# Config
with open('../config.yaml') as f:
    config = yaml.safe_load(f)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('Setup OK')
print(f'Proyecto: {config["project"]["name"]}')

## 1. Carga de datos

In [ ]:
# TODO Sprint 2: cargar datos procesados
# Descomenta cuando tengas los datos de HU1/HU2 completados

# df_dengue = pd.read_parquet('../data/processed/dengue_weekly_comuna.parquet')
# df_climate = pd.read_parquet('../data/processed/climate_weekly_comuna.parquet')
# gdf_comunas = gpd.read_file('../data/external/comunas_caba.geojson')

# --- Datos sintéticos para desarrollo (reemplazar con datos reales) ---
np.random.seed(42)
n_comunas = 15
n_years = 5
n_weeks = 52 * n_years

# Simular casos con estacionalidad (pico en semanas 5-15 = feb-abr)
weeks = np.arange(1, n_weeks + 1)
years = np.repeat(np.arange(2019, 2019 + n_years), 52)
epi_weeks = np.tile(np.arange(1, 53), n_years)

rows = []
for c in range(1, n_comunas + 1):
    base = np.random.poisson(2, n_weeks)
    seasonal = 15 * np.exp(-0.5 * ((epi_weeks - 8) / 4) ** 2)  # pico semana 8
    cases = (base + seasonal * np.random.uniform(0.5, 1.5, n_weeks)).astype(int)
    for i, (y, w, c_val) in enumerate(zip(years, epi_weeks, cases)):
        rows.append({'year': y, 'epi_week': w, 'comuna_id': c, 'confirmed_cases': c_val})

df_dengue = pd.DataFrame(rows)
print(f'Datos de dengue: {df_dengue.shape}')
print(f'Período: {df_dengue.year.min()} - {df_dengue.year.max()}')
print(f'Comunas: {df_dengue.comuna_id.nunique()}')
df_dengue.head()

## 2. Serie temporal de casos — CABA total

In [ ]:
# Casos totales por semana epidemiológica
df_total = df_dengue.groupby(['year', 'epi_week'])['confirmed_cases'].sum().reset_index()
df_total['date_label'] = df_total['year'].astype(str) + '-SE' + df_total['epi_week'].astype(str).str.zfill(2)

fig = px.line(
    df_total,
    x='date_label',
    y='confirmed_cases',
    title='Casos confirmados de dengue — CABA total (por semana epidemiológica)',
    labels={'confirmed_cases': 'Casos confirmados', 'date_label': 'Año-Semana epidemiológica'},
    color_discrete_sequence=['#E74C3C']
)

# Marcar temporada de riesgo (dic-abr = semanas 1-17 y 48-52)
# TODO: agregar sombreado de temporada

fig.update_layout(xaxis_tickangle=45, height=400)
fig.show()

print(f'\nEstadísticas globales:')
print(df_total['confirmed_cases'].describe())

## 3. Distribución por comuna

In [ ]:
# Casos totales por comuna (barplot)
df_by_comuna = df_dengue.groupby('comuna_id')['confirmed_cases'].sum().reset_index()
df_by_comuna.columns = ['comuna_id', 'total_cases']
df_by_comuna = df_by_comuna.sort_values('total_cases', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(
    df_by_comuna['comuna_id'].astype(str),
    df_by_comuna['total_cases'],
    color='#E74C3C',
    alpha=0.8,
    edgecolor='white'
)
ax.set_xlabel('Comuna')
ax.set_ylabel('Casos confirmados totales')
ax.set_title('Distribución de casos de dengue por comuna (total período)')
plt.tight_layout()
plt.savefig('../reports/figures/01_casos_por_comuna.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 comunas con más casos:')
print(df_by_comuna.head())

## 4. Heatmap: casos por semana y comuna

In [ ]:
# Pivot: filas = semanas, columnas = comunas
df_pivot = df_dengue.groupby(['epi_week', 'comuna_id'])['confirmed_cases'].mean().unstack()

fig, ax = plt.subplots(figsize=(15, 8))
sns.heatmap(
    df_pivot,
    cmap='YlOrRd',
    ax=ax,
    cbar_kws={'label': 'Casos promedio/semana'},
    linewidths=0.1
)
ax.set_xlabel('Comuna')
ax.set_ylabel('Semana epidemiológica')
ax.set_title('Heatmap: casos promedio de dengue por semana epidemiológica y comuna')

# Marcar temporada alta
ax.axhspan(0, 17, alpha=0.1, color='red', label='Temporada alta (SE 1-17)')
ax.axhspan(47, 52, alpha=0.1, color='red')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('../reports/figures/01_heatmap_semana_comuna.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlación clima-dengue

TODO: ejecutar cuando datos climáticos (HU2) estén disponibles

In [ ]:
# Placeholder - ejecutar con datos climáticos reales
# Variables a correlacionar con casos lag 1-4 semanas:
# - temp_mean, temp_max, temp_min
# - humidity
# - precipitation

print('TODO: calcular correlaciones de Pearson entre variables climáticas y casos')
print('Basado en Sebastianelli et al. (2024) y HU4 del plan de proyecto')
print()
print('Variables climáticas a analizar:')
for v in config['climate_variables']:
    print(f'  - {v["name"]}: {v["description"]}')

## 6. Hallazgos preliminares

*(Completar con datos reales en Sprint 3)*

### Patrones temporales
- [ ] Documentar semanas de pico histórico
- [ ] Identificar años epidémicos
- [ ] Calcular tasa de crecimiento semana a semana

### Patrones espaciales  
- [ ] Identificar comunas de mayor riesgo
- [ ] Analizar dispersión geográfica entre brotes
- [ ] Calcular autocorrelación espacial (índice de Moran)

### Variables climáticas
- [ ] Documentar lags con mayor correlación
- [ ] Identificar umbrales de temperatura/humedad asociados a brotes

### Limitaciones identificadas
- [ ] Período de subregistro en ciertas semanas/años
- [ ] Heterogeneidad en la notificación por comuna
- [ ] Cambio de criterios de diagnóstico en el período